In [1]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [2]:
sql = f"""
WITH target_devices AS (
  SELECT
    DeviceId,
    MIN(Time) AS first_takeaway_time
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND LOWER(EventLabelRaw) LIKE '%takeaway%'
  GROUP BY DeviceId
  ORDER BY first_takeaway_time
  LIMIT 20
)

SELECT pv.*
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN target_devices AS target
  ON pv.DeviceId = target.DeviceId
ORDER BY target.first_takeaway_time, pv.Time;
"""

#Execute query
df_bq = client.query(sql).result().to_dataframe()
df_bq

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,SessionId,DeviceId,Time,UserId,IP,EventAction,EventCategory,EventData,EventEntity,EventLabel,EventLabelRaw,EventSource,UserAgent,Product,MachineName,CollectTime,Platform
0,392510,61b7f680-6779-410e-a5db-0cf5911486ae,2026-09-15 00:00:00.400000+00:00,,49.130.128.36,or.takeaway.order,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;POIID:438899;sr:sr...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6);Lang:zh-HK,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-14 23:39:01+00:00,ios
1,392510,61b7f680-6779-410e-a5db-0cf5911486ae,2026-09-15 00:00:01.500000+00:00,,49.130.128.36,poi.emenu.photo,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;POIID:438899;Page:...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6);Lang:zh-HK,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-14 23:39:01+00:00,ios
2,392510,61b7f680-6779-410e-a5db-0cf5911486ae,2026-09-15 00:00:02.600000+00:00,,49.130.128.36,or.explore.reel.photo,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;poiId:438899;SrcPh...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6);Lang:zh-HK,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-14 23:39:01+00:00,ios
3,392510,61b7f680-6779-410e-a5db-0cf5911486ae,2026-09-15 00:00:06.200000+00:00,,49.130.128.36,or.explore.reel.photo,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;poiId:438899;photo...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6);Lang:zh-HK,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-14 23:39:01+00:00,ios
4,392510,61b7f680-6779-410e-a5db-0cf5911486ae,2026-09-15 00:00:11.300000+00:00,,49.130.128.36,or.explore.reel.photo,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;poiId:438899;photo...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6);Lang:zh-HK,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-14 23:39:01+00:00,ios
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4510,321134,4fcd6eb3-2652-4238-a7ef-ebfac13b7166,2026-09-15 23:47:05.400000+00:00,755b7c39-49f9-4b02-9bce-e42f6a2fb1e4,58.153.159.180,or.tm.book,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;tracking:uIwLRAw;P...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.5.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 23:26:01+00:00,ios
4511,321134,4fcd6eb3-2652-4238-a7ef-ebfac13b7166,2026-09-15 23:47:50.900000+00:00,755b7c39-49f9-4b02-9bce-e42f6a2fb1e4,58.153.159.180,or.orpay.offer.detail,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;POIID:0;offerId:0;sr:(null);sn:HK.TM....,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.5.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 23:27:01+00:00,ios
4512,321134,4fcd6eb3-2652-4238-a7ef-ebfac13b7166,2026-09-15 23:48:06.400000+00:00,755b7c39-49f9-4b02-9bce-e42f6a2fb1e4,58.153.159.180,or.coupon.getdetails,,,,"{'list': [{'item': {'List': 0, 'Param': 'CpnID...",CpnID:391116;CpnTp:1;sr:BookingDetail;CityID:0...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.5.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 23:27:01+00:00,ios
4513,321134,4fcd6eb3-2652-4238-a7ef-ebfac13b7166,2026-09-15 23:48:18.300000+00:00,,58.153.159.180,or.tm.succeed,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;tracking:uIwLRAw;P...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.5.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 23:27:01+00:00,ios
